<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 11 · Tree-Based Methods and Ensembles

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
We implement decision trees, Random Forests, and Gradient Boosted Trees on the feature panel and link them to portfolio diagnostics.

### Getting Help While Using Tree Models
- **Appendix C** lists scikit-learn APIs for trees/ensembles.
- **Chapter 7** covers feature engineering referenced here.
- **Chapter 8** reminds you how to evaluate strategy returns.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor  # 实际中很少会用GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
# from xgboost import XGBRegressor

## 1. Build Feature/Label Tables

We assemble the tree model inputs by computing rolling momentum and volatility features and aligning them with next‑day returns as labels.

In [3]:
prices = pd.read_csv(DATA_PATH,parse_dates=["Date"]).set_index("Date").sort_index().ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()
feature_components = {
    "momentum": prices.pct_change(20),
    "volatility": log_rets.rolling(20).std(),
}
features = pd.concat(feature_components, axis=1).dropna()  # features index为双层。
labels = log_rets.shift(-1)["AAPL"].rename("label")
common_index = features.index.intersection(labels.dropna().index)
features = features.loc[common_index]
labels = labels.loc[common_index]
features.head()

momentum                                                    \
                AAPL      NVDA       JPM       SPY       GLD       TLT   
Date                                                                     
2015-12-29 -0.080812  0.061773  0.005847 -0.000267  0.002747 -0.006448   
2015-12-30 -0.085393  0.019527 -0.015087 -0.016729 -0.008408 -0.020665   
2015-12-31 -0.094769  0.014127 -0.009451 -0.016524  0.007647 -0.016579   
2016-01-04 -0.085501 -0.001896 -0.026644 -0.016509  0.011105  0.018194   
2016-01-05 -0.137110 -0.025507 -0.054978 -0.033688 -0.008075  0.005255   

                               volatility                                \
              EURUSD   BTC-USD       AAPL      NVDA       JPM       SPY   
Date                                                                      
2015-12-29  0.028029  0.147519   0.015568  0.017025  0.017323  0.011755   
2015-12-30  0.024121  0.176922   0.015674  0.015769  0.017094  0.011634   
2015-12-31 -0.006308  0.198726   0.016000  0.015966  0.016905  0.011626   
2016-01-04 -0.011861  0.199545   0.016016  0.016503  0.018005  0.011625   
2016-01-05 -0.008919  0.189373   0.014068  0.014151  0.016321  0.010644   

                                                    
                 GLD       TLT    EURUSD   BTC-USD  
Date                                                
2015-12-29  0.011150  0.010643  0.008502  0.040484  
2015-12-30  0.011252  0.010140  0.008619  0.039318  
2015-12-31  0.010664  0.010212  0.005271  0.039106  
2016-01-04  0.010862  0.008180  0.005703  0.039103  
2016-01-05  0.009651  0.008033  0.005758  0.039186

## 2. Decision Tree Baseline

This baseline single tree provides an interpretable set of if–then rules that approximate the mapping from features to returns.

In [4]:
split = int(len(features) * 0.7)
X_train, X_test = features.iloc[:split], features.iloc[split:]
y_train, y_test = labels.iloc[:split], labels.iloc[split:]

# 树深度：3~5最稳，4是黄金中间值
tree = DecisionTreeRegressor(max_depth=4, random_state=0)
# 其他参数：min_samples_split=20, # 至少20个样本才允许分裂，防过拟合。
# min_samples_leaf=10, # 叶子节点最少10个样本

tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)
mean_squared_error(y_test, tree_pred)

0.0002680271582834564

## 3. Random Forest and Gradient Boosting

We then fit Random Forest and Gradient Boosted Trees to reduce variance and capture more subtle nonlinearities than a lone tree can.

In [5]:
rf = RandomForestRegressor(
    n_estimators=300,      # 300棵树
    max_depth=6,           # 每棵树的深度
    min_samples_leaf=50,   # 叶子节点50个样本
    random_state=0,
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
gbt = GradientBoostingRegressor(random_state=0)
gbt.fit(X_train, y_train)
gbt_pred = gbt.predict(X_test)
pd.Series(
    {
        "tree_mse": mean_squared_error(y_test, tree_pred),
        "rf_mse": mean_squared_error(y_test, rf_pred),
        "gbt_mse": mean_squared_error(y_test, gbt_pred),
    }
)

,0
tree_mse,0.000268
rf_mse,0.000270
gbt_mse,0.000309


### 3.1 Feature Importance Comparison

Using built‑in importance scores, we compare how RF and GBT rank the same set of features, which is a first step toward explainability.

In [7]:
importances = pd.DataFrame(
    {
        "RF": rf.feature_importances_,
        "GBT": gbt.feature_importances_,
        # 每次训练、换随机种子、换参数，feature_importances 结果都会变一点，但整体排序基本稳定。
        # 用来筛选因子，解释模型。
    },
    index=features.columns,
)
importances

RF       GBT
momentum   AAPL     0.097636  0.092800
           NVDA     0.066095  0.064320
           JPM      0.072830  0.137298
           SPY      0.068176  0.129758
           GLD      0.032463  0.042455
           TLT      0.106592  0.126854
           EURUSD   0.063896  0.048315
           BTC-USD  0.040648  0.084135
volatility AAPL     0.077405  0.072265
           NVDA     0.080799  0.037168
           JPM      0.068912  0.027307
           SPY      0.063534  0.023656
           GLD      0.045967  0.040522
           TLT      0.042114  0.038467
           EURUSD   0.028201  0.018527
           BTC-USD  0.044731  0.016152

### Performance Helper
We reuse the Chapter 8 metric function to translate predictions into stats.

In [8]:
def performance_stats(returns: pd.Series, risk_free: float = 0.02):
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - risk_free) / ann_vol if ann_vol > 0 else np.nan
    wealth = (1 + returns).cumprod()
    max_dd = (wealth / wealth.cummax() - 1).min()
    return pd.Series(
        {
            "annualized_return": ann_ret,
            "annualized_vol": ann_vol,
            "sharpe": sharpe,
            "max_drawdown": max_dd,
        }
    )

## 4. Portfolio Signal from Random Forest Predictions

Finally, we convert Random Forest predictions into cross‑sectional ranks and compute portfolio returns to see whether the model adds value.

In [9]:
signal = pd.Series(rf_pred, index=X_test.index).rank(pct=True) - 0.5
# rank(pct=True)是将预测值转化为（0，1）的percentage -> 将模型预测转化为【截面排名】。
# -0.5将（0，1）——> (-0.5, +0.5)排名前50%的做多，排名后50%的做空。将截面排名转化为【标准化多空信号】。
strategy_returns = signal * y_test
performance_stats(strategy_returns)

,0
annualized_return,0.022597
annualized_vol,0.081268
sharpe,0.031957
max_drawdown,-0.101748


## 5. Exercises
### Exercise 1 – Hyperparameter Sweep
Evaluate RF/GBT performance over grids of depth and learning rate.
<details><summary>Hint</summary>
Use <code>RandomizedSearchCV</code> with custom scoring (e.g., negative MSE).
</details>

### Exercise 2 – Permutation Importance
Implement permutation importance for the RF model and compare to built-in feature importances.
<details><summary>Hint</summary>
Use <code>sklearn.inspection.permutation_importance</code> with <code>n_repeats=20</code>.
</details>

### Exercise 3 – Turnover Control
Smooth the RF signal using an exponential moving average before computing strategy returns.
<details><summary>Hint</summary>
Apply <code>.ewm(alpha=0.2).mean()</code> to the signal series.
</details>


## 6. Takeaways for Chapter 11
- Tree ensembles offer nonlinear pattern capture with minimal feature engineering.
- Comparing tree/RF/GBT metrics ensures we balance accuracy with interpretability.
- Translating ensemble scores into portfolio signals exposes turnover and stability considerations.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">